# 00 — Verify your setup

Run every cell in this notebook (Kernel → Restart & Run All). Each section is one discrete check. You'll see ✅ for things that work, ❌ for things that don't — and every failure comes with a concrete next step.

If every check is green, you're ready for `chapter-5/01_advil_moment.ipynb`.

> **Where to run this:** inside the Jupyter Lab served by the book's Docker stack (URL is `http://localhost:8888`). If you opened this in a host-installed Jupyter, `import aips` will fail. See `tutorials/SETUP.md` for setup steps.

In [ ]:
# Small helper used by every check cell.
# It prints a single line with a verdict and (on failure) a fix-it hint.

results = []

def check(name: str, ok: bool, detail: str = "", fix: str = "") -> bool:
    icon = "✅" if ok else "❌"
    line = f"{icon}  {name}"
    if detail:
        line += f"  —  {detail}"
    print(line)
    if not ok and fix:
        print(f"     ↳ fix: {fix}")
    results.append((name, ok))
    return ok

## Check 1 — The `aips` package is importable

This is the most common point of failure. If this cell fails, you're almost certainly running Jupyter outside the book's Docker container.

In [ ]:
try:
    import aips
    from aips import get_engine, get_skg
    check(
        "aips package importable",
        True,
        detail=f"version={getattr(aips, '__version__', 'unknown')}",
    )
except ImportError as e:
    check(
        "aips package importable",
        False,
        detail=str(e),
        fix="Open this notebook via http://localhost:8888 (the book's Docker Jupyter), not a host-installed Jupyter.",
    )

## Check 2 — The search engine is reachable

The book defaults to Apache Solr. From inside the Docker network it's reached as `aips-solr:8983`; `aips.get_engine()` handles that for you.

In [ ]:
engine = None
try:
    engine = get_engine()
    engine_name = engine.name
    # A no-op call that forces the engine to respond. We list collections.
    # If the engine isn't up, this raises.
    _ = engine.get_collection("health")  # may not exist yet; we test that below
    check(
        "Engine reachable",
        True,
        detail=f"engine={engine_name}",
    )
except Exception as e:
    check(
        "Engine reachable",
        False,
        detail=str(e),
        fix="Is the Docker stack up? Run `docker compose up` from the repo root and wait until Solr logs 'Registered new searcher'.",
    )

## Check 3 — The `health` collection has documents

This dataset (Stack Exchange health forum posts) is what Tutorial 1 traverses to find `advil → motrin, aleve, ibuprofen`.

In [ ]:
try:
    health = engine.get_collection("health")
    # `query=*` matches everything; we just want a count.
    response = health.search(query="*", limit=0)
    # The aips collection wrapper returns dict-style results; the
    # 'total' / 'num_found' key may vary. Try both.
    total = response.get("total") or response.get("num_found") or len(response.get("docs", []))
    if total and total > 0:
        check("health collection populated", True, detail=f"{total:,} documents")
    else:
        check(
            "health collection populated",
            False,
            detail="0 documents",
            fix="Open chapters/ch5/ in Jupyter and run the health-dataset setup notebook to index the data.",
        )
except Exception as e:
    check(
        "health collection populated",
        False,
        detail=str(e),
        fix="Collection probably doesn't exist yet. Open chapters/ch5/ and run the health-dataset setup notebook.",
    )

## Check 4 — The `stackexchange` (scifi) collection has documents

Used by Tutorial 2 (vibranium query expansion) and Tutorial 3 (Jean Grey typed-edge traversal).

In [ ]:
# The book uses one combined 'stackexchange' collection for several
# scifi/cooking/etc. sub-forums. If your setup separated scifi out,
# adjust the collection name.
for candidate in ("stackexchange", "scifi"):
    try:
        coll = engine.get_collection(candidate)
        response = coll.search(query="*", limit=0)
        total = response.get("total") or response.get("num_found") or 0
        if total and total > 0:
            check(f"{candidate} collection populated", True, detail=f"{total:,} documents")
            break
    except Exception:
        continue
else:
    check(
        "scifi/stackexchange collection populated",
        False,
        detail="neither 'stackexchange' nor 'scifi' returned docs",
        fix="Open chapters/ch5/ and run the stackexchange-dataset setup notebook.",
    )

## Check 5 — An end-to-end SKG traversal actually works

This is the real smoke test. If we can run a traversal on `advil` and get drug-like neighbors back, the entire pipeline (engine + faceting + RelatednessAgg + aips abstractions) is wired correctly.

In [ ]:
try:
    health_skg = get_skg(engine.get_collection("health"))
    traversal = health_skg.traverse(
        {"field": "body", "values": ["advil"]},
        {"field": "body", "min_occurrences": 2, "limit": 5},
    )

    # Pull the related-term names out of the response. The shape is:
    # traversal['graph'][0]['values']['advil']['traversals'][0]['values']
    raw = traversal["graph"][0]["values"]["advil"]["traversals"][0]["values"]
    related_terms = list(raw.keys())

    if len(related_terms) >= 3:
        check(
            "SKG traversal returns related terms",
            True,
            detail=f"top neighbors: {', '.join(related_terms[:5])}",
        )
    else:
        check(
            "SKG traversal returns related terms",
            False,
            detail=f"only got {len(related_terms)} neighbors",
            fix="The health collection may be under-populated. Re-run the health setup notebook.",
        )
except Exception as e:
    check(
        "SKG traversal returns related terms",
        False,
        detail=str(e),
        fix="Check that prior checks pass. If they do, the SKG implementation may not be available for your chosen engine (Solr is the default and supports it natively).",
    )

## Check 6 — Visualization dependencies are present

The tutorials use `matplotlib` for bar charts and `networkx` for the graph view. Both ship with the book's Jupyter image, but it doesn't hurt to confirm.

In [ ]:
missing = []
for mod in ("matplotlib", "networkx"):
    try:
        __import__(mod)
    except ImportError:
        missing.append(mod)

if not missing:
    check("Visualization libraries installed", True, detail="matplotlib + networkx OK")
else:
    check(
        "Visualization libraries installed",
        False,
        detail=f"missing: {', '.join(missing)}",
        fix=f"Run `!pip install {' '.join(missing)}` in a cell, then re-run.",
    )

## Check 7 — Our `skg_viz` helper is importable

This confirms `tutorials/chapter-5/skg_viz.py` is visible to Python. If this fails, the tutorial notebook will fail later — fix it now.

In [ ]:
import sys
import os

# Make the chapter-5 folder importable from this notebook.
ch5_dir = os.path.abspath(os.path.join(os.getcwd(), "chapter-5"))
if ch5_dir not in sys.path:
    sys.path.insert(0, ch5_dir)

try:
    from skg_viz import plot_relatedness_bars, draw_graph  # noqa: F401
    check("skg_viz helper importable", True, detail=f"found in {ch5_dir}")
except ImportError as e:
    check(
        "skg_viz helper importable",
        False,
        detail=str(e),
        fix="Make sure tutorials/chapter-5/skg_viz.py exists alongside the notebooks.",
    )

## Summary

Final scoreboard. If you see all green, you're ready.

In [ ]:
passed = sum(1 for _, ok in results if ok)
total = len(results)
print(f"\n{'='*60}")
print(f"  {passed} / {total} checks passing")
print(f"{'='*60}\n")

for name, ok in results:
    icon = "✅" if ok else "❌"
    print(f"  {icon}  {name}")

print()
if passed == total:
    print("\ud83c\udf89  You're set. Open tutorials/chapter-5/01_advil_moment.ipynb next.")
else:
    print("⚠\ufe0f   Scroll back up to the failed checks for their fix-it lines.")